### This is an experiment for new image-text retrieval web ###
NOTES:
- This experiment uses Apple's version of CLIP, namely "DFN5B CLIP ViT H14"
- The following part is to create a vector collection on Zilliz Vector Database, with the help of Milvus

**SETUP**

In [1]:
import torch
from tqdm import tqdm
from pprint import pprint
from pymilvus import MilvusClient, DataType, CollectionSchema, FieldSchema

**DATASET LOAD**

In [2]:
full_dataset = torch.load('aic_2025-wfeat.pt')
data_insert = list(full_dataset.values())  # Data to be inserted into Zilliz Cloud is values of dataset dictionary

In [5]:
pprint(data_insert[123456])

{'answer_key': 'L26_V381-102400',
 'frame_id': '068',
 'frame_order': '2560',
 'img_key': 'kfs/Keyframes_L26/L26_V381/068.jpg',
 'publish_date': '30/01/2021',
 'time_order': '102400',
 'vector': [0.018310420215129852,
            0.0036270434502512217,
            0.007783419452607632,
            0.006225244142115116,
            0.011444011703133583,
            0.014016118831932545,
            -0.006363168824464083,
            0.009259585291147232,
            -0.002974697621539235,
            -0.016864074394106865,
            -0.02980661951005459,
            0.00722053786739707,
            0.030149566009640694,
            0.01772889867424965,
            0.007593307178467512,
            -0.006624107249081135,
            0.022246861830353737,
            -0.028613757342100143,
            -0.013665716163814068,
            -0.0026559799443930387,
            0.013911743648350239,
            0.030835462734103203,
            -0.0035636727698147297,
            -0.0113843688

**DATASET TWEAKS**

In [14]:
for entity in list(full_dataset.values()):
    img_key = entity['img_key']
    separations = img_key.split('/')
    vid_key = f'vids/{separations[1].replace('Keyframes', 'Videos')}/{separations[2]}.mp4'
    entity['vid_key'] = vid_key

**CONNECT TO CLIENT**

In [3]:
client = MilvusClient(
    uri='https://in03-cbdb10d1d199984.serverless.aws-eu-central-1.cloud.zilliz.com',
    token='6305ff67695e59bf211ca717cb68f74f7367ccfd56195824ba6aad7c07f5924cc6c9da8aadec4354aedc193a997225494903a9d7'
)

**CREATE COLLECTION'S SCHEMA**

In [15]:
#Schema build
id = FieldSchema(
    'id',
    dtype=DataType.INT64,
    is_primary=True,
    auto_id=True
)
vector = FieldSchema(
    'vector',
    dtype=DataType.FLOAT_VECTOR,
    dim=1024,
    description='Feature vectors obtained from DFN5B-CLIP-ViT-H14 computation.'
)
img_link = FieldSchema(
    'img_link',
    dtype=DataType.VARCHAR,
    max_length=1024,
    description='Image link acquired from GCS. For main display'
)

vid_link = FieldSchema(
    'vid_link',
    dtype=DataType.VARCHAR,
    max_length=1024,
    description='Video link acquired from GCS. For revalidation'
)

video_id = FieldSchema(
    'video_id',
    dtype=DataType.VARCHAR,
    max_length=32,
    is_partition_key=True,
    description='The ID of the orignal video, where the frame belongs to.'
)

frame_id = FieldSchema(
    'frame_id',
    dtype=DataType.VARCHAR,
    max_length=16,
    description='The ID as the name of the image given by contest organizers.'
)

time_order = FieldSchema(
    'time_order',
    dtype=DataType.VARCHAR,
    max_length=16,
    description='The time where the image appears in the original video, in [ms].'
)

frame_order = FieldSchema(
    'frame_order',
    dtype=DataType.VARCHAR,
    max_length=16,
    description='The n-th frame of the video.'
)

fps = FieldSchema(
    'fps',
    dtype=DataType.VARCHAR,
    max_length=16,
    description='The frame rate of the video'
)

answer_key = FieldSchema(
    'answer_key',
    dtype=DataType.VARCHAR,
    max_length=256,
    description='The answer key to be sent via API calls. Only used in Finals.'
)

youtube_link = FieldSchema(
    'youtube_link',
    dtype=DataType.VARCHAR,
    max_length=1024,
    description='The YouTube link of the video where this image belong to.'
)

publish_date = FieldSchema(
    'publish_date',
    dtype=DataType.VARCHAR,
    max_length=32,
    description='The publish date of the original video. Mostly untouched.'
)

schema = CollectionSchema(
    fields=[
        id,
        vector,
        img_link,
        vid_link,
        video_id,
        frame_id,
        time_order,
        frame_order,
        fps,
        answer_key,
        youtube_link,
        publish_date
    ],
    description='This schema gives most of the essential details for querying and filtering images. Made specifically for AIC.'
)

**PREPARE NEW INDEX**

In [16]:
index_params = MilvusClient.prepare_index_params()

index_params.add_index(
    field_name='vector',
    metric_type='COSINE',
    index_type='HNSW',
    index_name='vector_index',
    params={
        'M': 32,
        'efConstruction': 200
    }
)

**CREATE NEW COLLECTION**

In [17]:
if client.has_collection(collection_name="aic_2025"):
    client.drop_collection(collection_name="aic_2025")
client.create_collection(
    collection_name="aic_2025",
    schema=schema,
    index_params=index_params,
    enable_dynamic_field=True,
    consistency_level='Eventually'
)

**BATCHING**

In [7]:
BATCH_SIZE = 12288
batches = []
for i in range(0, len(data_insert), BATCH_SIZE):
    batches.append(data_insert[i:i + BATCH_SIZE] if i < len(data_insert) else data_insert[i:len(data_insert)])

print(f'Done batching. Batch count: {len(batches)}')

Done batching. Batch count: 32


**INSERT VECTORS TO COLLECTION**

In [18]:
with tqdm(batches, desc='Inserting batches to vector database') as t:
    for batch in t:
        client.insert(
            'aic_2025',
            data=batch
        )

        elapsed = t.format_dict['elapsed']
        elapsed_str = t.format_interval(elapsed)

Inserting batches to vector database: 100%|██████████| 32/32 [14:42<00:00, 27.57s/it]


**DEBUGGING**

In [15]:
torch.save(full_dataset,'dataset-aic2025-with-feature-b1.pt')